# Simulate a synthetic binary-star population

This notebook generates the dataset consumed by [`docs/tutorials/case-studies/population-binary-fraction.ipynb`](../case-studies/population-binary-fraction.ipynb). The output is a single HDF5 file at `synthetic_binary_population/population.h5`.

$$
f_\mathrm{close} = 0.40
$$

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

In [ ]:
import pathlib

import h5py
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import truncnorm
from unxt import Q, ustrip
import numpyro.distributions as dist

from harv import RVData
from harv.kepler.constants import G
from harv.kepler.orbits import rv_at_times

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

## Population parameters

In [ ]:
SEED = 0

N_STARS = 100
BINARY_FRACTION = 0.40  # raw simulation fraction (companion vs. no companion)
N_EPOCHS = 10
BASELINE = Q(8 * 365, "day")
RV_ERR = Q(1.0, "km/s")

# True eccentricity distribution (TruncatedNormal on [0, 1]).
# Section 3 of the case study should recover these.
ECC_MEAN, ECC_STD = 0.4, 0.15

# Period range for the simulated binaries.
P_MIN_SIM = Q(5.0, "day")
P_MAX_SIM = Q(1000.0, "day")
logP_MEAN = 5.03  # Raghavan+2010
logP_STD = 2.28

# Companion mass range (log-uniform).  Spans the 0.1 Msun cut so the close
# binary fraction differs from the raw binary fraction.
M2_MIN = Q(0.1, "Msun")
M2_MAX = Q(0.9, "Msun")

# Primary mass: Gaussian around 1 Msun.
M1_MEAN = Q(1.0, "Msun")
M1_STD = Q(0.05, "Msun")

# Systemic velocity scale.
SIGMA_VSYS = Q(50.0, "km/s")

# Close-binary cuts (must match the case-study notebook).
P_MIN_CUT = Q(1.0, "day")
P_MAX_CUT = Q(1000.0, "day")
M2_CUT = Q(0.1, "Msun")

OUT_DIR = pathlib.Path("synthetic_binary_population")
OUT_DIR.mkdir(exist_ok=True)
OUT_PATH = OUT_DIR / "population.h5"

## Draw the shared time grid

A single sorted uniform draw over `BASELINE` defines the time grid for every star. Using the same grid across stars is what lets the rejection sampler reuse its JIT cache in Section 2 of the case study.

In [ ]:
rng = np.random.default_rng(SEED)

times_d = np.sort(rng.uniform(0.0, ustrip("day", BASELINE), size=N_EPOCHS))
times = Q(times_d, "day")

print(f"N_epochs = {N_EPOCHS}, baseline = {BASELINE}")
print(f"First / last epoch: {times_d[0]:.1f} d, {times_d[-1]:.1f} d")

## Draw per-star population truths

We always draw the full set of orbital parameters for every star; for the 60 "single" stars these draws are simply unused (we record them as `nan`s in the truth dict).

In [ ]:
def K_from_physical(
    M1: "Q", M2: "Q", period: "Q", e: np.ndarray, sini: np.ndarray
) -> "Q":
    r"""K = (2 pi G / P)^{1/3} * (M2 sin i) / ((M1 + M2)^{2/3} * sqrt(1 - e^2))."""
    one = Q(np.ones_like(np.asarray(e)), "")
    pref = (Q(2.0 * np.pi, "") * G / period) ** (1.0 / 3.0)
    num = M2 * (sini * one)
    den = (M1 + M2) ** (2.0 / 3.0) * (one - (e * one) ** 2) ** 0.5
    return pref * num / den


# 1) Decide which stars are binaries (40% of N_stars rounded to N_binaries=40).
n_binaries = int(round(BINARY_FRACTION * N_STARS))
is_binary = np.zeros(N_STARS, dtype=bool)
binary_idx = rng.choice(N_STARS, size=n_binaries, replace=False)
is_binary[binary_idx] = True
print(f"binaries: {is_binary.sum()} / {N_STARS}")

# 2) Always draw the full set of orbital parameters; mask later for singles.
M1_msun = rng.normal(ustrip("Msun", M1_MEAN), ustrip("Msun", M1_STD), size=N_STARS)
M2_msun = np.exp(
    rng.uniform(
        np.log(ustrip("Msun", M2_MIN)),
        np.log(ustrip("Msun", M2_MAX)),
        size=N_STARS,
    )
)
# period_d = np.exp(
#     rng.uniform(
#         np.log(ustrip("day", P_MIN_SIM)),
#         np.log(ustrip("day", P_MAX_SIM)),
#         size=N_STARS,
#     )
# )
log_P_d = dist.TruncatedNormal(
    logP_MEAN,
    logP_STD,
    low=np.log10(ustrip("day", P_MIN_SIM)),
    high=np.log10(ustrip("day", P_MAX_SIM)),
).sample(jr.key(42), (N_STARS,))
period_d = np.array(10**log_P_d)

a_trunc = (0.0 - ECC_MEAN) / ECC_STD
b_trunc = (1.0 - ECC_MEAN) / ECC_STD
ecc = truncnorm.rvs(
    a_trunc,
    b_trunc,
    loc=ECC_MEAN,
    scale=ECC_STD,
    size=N_STARS,
    random_state=rng,
)
cos_i = rng.uniform(-1.0, 1.0, size=N_STARS)
sin_i = np.sqrt(1.0 - cos_i**2)
arg_peri_rad = rng.uniform(0.0, 2 * np.pi, size=N_STARS)
t_peri_d = rng.uniform(0.0, period_d, size=N_STARS)
v_sys_kms = rng.normal(0.0, ustrip("km/s", SIGMA_VSYS), size=N_STARS)

# 3) Compute K (km/s) for every star's orbital draw.
#    ``ustrip`` returns a JAX array; convert to numpy so the masking step
#    below uses regular in-place assignment.
K_kms = np.asarray(
    ustrip(
        "km/s",
        K_from_physical(
            Q(M1_msun, "Msun"),
            Q(M2_msun, "Msun"),
            Q(period_d, "day"),
            ecc,
            sin_i,
        ),
    )
).astype(np.float64)

# 4) For singles, mask the orbital parameters to NaN so downstream consumers
#    don't accidentally use them.
single = ~is_binary
M2_msun[single] = np.nan
period_d[single] = np.nan
ecc[single] = np.nan
sin_i[single] = np.nan
arg_peri_rad[single] = np.nan
t_peri_d[single] = np.nan
K_kms[single] = 0.0  # zero amplitude -> flat RV under rv_at_times
print(
    f"K (binaries) range: {K_kms[is_binary].min():.2f} - {K_kms[is_binary].max():.2f} km/s"
)

## Compute the radial-velocity time series for every star

In [ ]:
rv_kms = np.empty((N_STARS, N_EPOCHS), dtype=np.float32)
rv_err_kms = np.full((N_STARS, N_EPOCHS), ustrip("km/s", RV_ERR), dtype=np.float32)

for n in range(N_STARS):
    if is_binary[n]:
        rv_clean = rv_at_times(
            times,
            period=Q(float(period_d[n]), "day"),
            eccentricity=float(ecc[n]),
            t_peri=Q(float(t_peri_d[n]), "day"),
            arg_peri=Q(float(arg_peri_rad[n]), "rad"),
            rv_semiamp=Q(float(K_kms[n]), "km/s"),
            v_sys=Q(float(v_sys_kms[n]), "km/s"),
        )
        rv_kms[n] = np.asarray(ustrip("km/s", rv_clean), dtype=np.float32)
    else:
        # Single star -- flat RV at v_sys.
        rv_kms[n] = float(v_sys_kms[n])

# Add Gaussian noise.
rv_kms = rv_kms + rng.normal(scale=ustrip("km/s", RV_ERR), size=rv_kms.shape).astype(
    np.float32
)
print(f"rv shape: {rv_kms.shape}, dtype: {rv_kms.dtype}")

## Quick look at a few simulated stars

In [ ]:
show_binaries = rng.choice(np.where(is_binary)[0], size=3, replace=False)
show_singles = rng.choice(np.where(~is_binary)[0], size=3, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(11, 5.5), sharex=True)
t = np.asarray(ustrip("day", times))
for ax, idx in zip(axes[0], show_binaries):
    ax.errorbar(t, rv_kms[idx], yerr=rv_err_kms[idx], fmt="o", ms=3, lw=0.8)
    P = period_d[idx]
    K = K_kms[idx]
    ax.set_title(f"binary {idx}: P={P:.1f}d, K={K:.1f}km/s", fontsize=9)
for ax, idx in zip(axes[1], show_singles):
    ax.errorbar(t, rv_kms[idx], yerr=rv_err_kms[idx], fmt="o", ms=3, lw=0.8, color="C2")
    ax.set_title(f"single {idx}: v_sys={v_sys_kms[idx]:.1f}km/s", fontsize=9)
for ax in axes[-1]:
    ax.set_xlabel("time [day]")
for ax in axes[:, 0]:
    ax.set_ylabel("RV [km/s]")
fig.tight_layout()

## Compute the "truth" close binary fraction

Following the case-study definition: fraction of stars satisfying both $P \in (P_\min^{\rm cut}, P_\max^{\rm cut})$ and $M_2 > M_{2,\rm cut}$. Singles have NaN $P$ and $M_2$, so they fail both cuts.


In [ ]:
P_min_d_cut = ustrip("day", P_MIN_CUT)
P_max_d_cut = ustrip("day", P_MAX_CUT)
M2_cut_msun = ustrip("Msun", M2_CUT)

with np.errstate(invalid="ignore"):
    in_P_cut = (period_d > P_min_d_cut) & (period_d < P_max_d_cut)
    in_M2_cut = M2_msun > M2_cut_msun
    is_close_binary = in_P_cut & in_M2_cut

binary_fraction_true = float(is_close_binary.mean())
print(f"raw simulation fraction:        {BINARY_FRACTION:.3f}")
print(f"close binary fraction (truth):  {binary_fraction_true:.3f}")
print(f"  ({int(is_close_binary.sum())} of {N_STARS} stars pass both cuts)")

## Save the population file

Schema written to HDF5:

```
/time         (N_stars, N_epochs)   float32   day        -- shared grid replicated per star
/rv           (N_stars, N_epochs)   float32   km/s
/rv_err       (N_stars, N_epochs)   float32   km/s
/truths/M1            (N_stars,)    float32   Msun
/truths/M2            (N_stars,)    float32   Msun        -- NaN for singles
/truths/period        (N_stars,)    float32   day         -- NaN for singles
/truths/eccentricity  (N_stars,)    float32               -- NaN for singles
/truths/sini          (N_stars,)    float32               -- NaN for singles
/truths/arg_peri      (N_stars,)    float32   rad         -- NaN for singles
/truths/t_peri        (N_stars,)    float32   day         -- NaN for singles
/truths/K             (N_stars,)    float32   km/s        -- 0   for singles
/truths/v_sys         (N_stars,)    float32   km/s
/truths/is_binary     (N_stars,)    bool
/truths/is_close_binary (N_stars,)  bool
```

Population-level scalars are stored as `/truths` group attributes: `simulation_binary_fraction`, `binary_fraction_true`, `ecc_alpha_true`, `ecc_beta_true`, the population-level cuts, and the RNG `seed`.


In [ ]:
with h5py.File(OUT_PATH, "w") as f:
    f.create_dataset(
        "time", data=np.broadcast_to(t, (N_STARS, N_EPOCHS)).astype(np.float32)
    )
    f.create_dataset("rv", data=rv_kms.astype(np.float32))
    f.create_dataset("rv_err", data=rv_err_kms.astype(np.float32))
    f["time"].attrs["unit"] = "day"
    f["rv"].attrs["unit"] = "km/s"
    f["rv_err"].attrs["unit"] = "km/s"

    g = f.create_group("truths")
    for name, arr, unit in [
        ("M1", M1_msun, "Msun"),
        ("M2", M2_msun, "Msun"),
        ("period", period_d, "day"),
        ("eccentricity", ecc, ""),
        ("sini", sin_i, ""),
        ("arg_peri", arg_peri_rad, "rad"),
        ("t_peri", t_peri_d, "day"),
        ("K", K_kms, "km/s"),
        ("v_sys", v_sys_kms, "km/s"),
    ]:
        ds = g.create_dataset(name, data=np.asarray(arr, dtype=np.float32))
        ds.attrs["unit"] = unit
    g.create_dataset("is_binary", data=is_binary)
    g.create_dataset("is_close_binary", data=is_close_binary)

    g.attrs["simulation_binary_fraction"] = float(BINARY_FRACTION)
    g.attrs["binary_fraction_true"] = binary_fraction_true
    g.attrs["ecc_mean_true"] = float(ECC_MEAN)
    g.attrs["ecc_std_true"] = float(ECC_STD)
    g.attrs["P_min_cut_day"] = float(P_min_d_cut)
    g.attrs["P_max_cut_day"] = float(P_max_d_cut)
    g.attrs["log10P_mean_true"] = float(logP_MEAN)
    g.attrs["log10P_std_true"] = float(logP_STD)
    g.attrs["M2_cut_msun"] = float(M2_cut_msun)
    g.attrs["seed"] = SEED

print(f"wrote {OUT_PATH} ({OUT_PATH.stat().st_size / 1024:.1f} KB)")